<a href="https://colab.research.google.com/github/flipiwolker-alt/cv-video-analytics/blob/main/pz_5_yolo_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ПЗ 5 — Детекция объектов YOLO

Прогоняем кадры из Drive через YOLOv8, сохраняем bounding boxes и классы.

In [ ]:
!pip install ultralytics opencv-python-headless pandas tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')

Mounted at /content/drive
кадров: 438


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

CONF = 0.4
rows = []

for fname in tqdm(frames, desc='yolo'):
    res = model(f'{FRAMES_DIR}/{fname}', conf=CONF, verbose=False)
    for r in res:
        for box in r.boxes:
            rows.append({
                'frame':     fname,
                'frame_num': int(fname.split('_')[1].split('.')[0]),
                'class':     model.names[int(box.cls[0])],
                'conf':      round(float(box.conf[0]), 3),
                'x1': round(float(box.xyxy[0][0])),
                'y1': round(float(box.xyxy[0][1])),
                'x2': round(float(box.xyxy[0][2])),
                'y2': round(float(box.xyxy[0][3])),
            })

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/yolo_detections.csv', index=False)
print(f'детекций: {len(df)}')
df.head(10)

yolo:   0%|          | 0/438 [00:00<?, ?it/s]

детекций: 556


,frame,frame_num,class,conf,x1,y1,x2,y2
0,frame_000030.jpg,30,person,0.902,91,5,1307,1058
1,frame_000150.jpg,150,person,0.916,670,109,1515,1067
2,frame_000150.jpg,150,cell phone,0.693,927,376,1008,527
3,frame_000180.jpg,180,person,0.795,580,4,1596,1065
4,frame_000180.jpg,180,tie,0.606,870,372,1049,1060
5,frame_000210.jpg,210,person,0.940,449,98,1340,1068
6,frame_000210.jpg,210,cell phone,0.764,936,381,1011,541
7,frame_000210.jpg,210,person,0.536,1342,38,1920,1066
8,frame_000240.jpg,240,person,0.939,384,6,1304,1070
9,frame_000270.jpg,270,person,0.847,2,8,1439,1069


In [ ]:
print(df.groupby('class')['conf'].agg(['count', 'mean']).sort_values('count', ascending=False))

              count      mean
class                        
person          489  0.823697
tie              20  0.518500
cell phone       18  0.649333
chair            14  0.571571
potted plant      2  0.694000
donut             2  0.522000
bird              2  0.487500
wine glass        2  0.653000
cup               1  0.463000
bottle            1  0.438000
cat               1  0.588000
dining table      1  0.530000
teddy bear        1  0.594000
umbrella          1  0.590000
vase              1  0.406000


In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

img = cv2.imread(f'{FRAMES_DIR}/{frames[0]}')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img_rgb)

for _, row in df[df['frame'] == frames[0]].iterrows():
    rect = Rectangle(
        (row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1,
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(row.x1, row.y1 - 5, f"{row['class']} {row.conf:.2f}", color='red', fontsize=9)

ax.axis('off')
plt.tight_layout()
plt.show()